# UMAP: Uniform Manifold Approximation and Projection

**Topic:** Unsupervised Learning — Dimensionality Reduction

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, FloatSlider, Output, HBox, VBox
from IPython.display import display, clear_output, Markdown
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print('umap-learn not installed. Run: pip install umap-learn')

if UMAP_AVAILABLE:
    # Warm up UMAP's JIT compilation on a tiny dummy array so the first
    # real call (in the widget below) isn't penalized by one-time compile cost.
    umap.UMAP(n_neighbors=5, n_components=2).fit_transform(np.random.rand(50, 4))

from tkh_utils import PALETTE, FONT, base_layout, load_california_housing


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** how UMAP differs from t-SNE in the type of structure it preserves
- **Explain** the role of n_neighbors and min_dist in controlling the UMAP layout
- **Interpret** a UMAP plot and compare it with PCA and t-SNE on the same data

> **Tip:** Install umap-learn if you haven't already (`pip install umap-learn`). Then try n_neighbors at 5 vs 50 and watch the local vs global structure change — similar to t-SNE's perplexity, but UMAP handles global distances more faithfully.

---
## How we got here

In **[08_tsne.ipynb](08_tsne.ipynb)** you learned t-SNE's strengths (excellent local cluster separation) and its key limitations (stochastic, slow, no new-point projection, inter-cluster distances meaningless).

UMAP, introduced by McInnes et al. in 2018, addresses most of those limitations. It is faster, more scalable, preserves global structure better, and can project new points without rerunning the algorithm. The trade-off is a more mathematically complex foundation.

---
## Why this matters for data science

UMAP has largely replaced t-SNE as the preferred dimensionality reduction visualization tool in most applied ML work. It is:
- 5-10x faster than t-SNE for medium datasets
- Scalable to 1M+ samples with approximate nearest neighbor search
- Better at preserving global structure (cluster distances are more meaningful)
- Able to project new data points (transform-only mode)

It is now the default choice for bioinformatics, NLP embedding visualization, and any exploratory high-dimensional analysis.

In **[ml_concepts/11_the_curse_of_dimensionality.ipynb](../ml_concepts/11_the_curse_of_dimensionality.ipynb)** you learned why distance-based methods degrade in high dimensions. UMAP is built explicitly around preserving a graph of true neighbors, which is why it holds up better than raw distance calculations as dimensionality grows.

---
## Where it sits on the spectrum

Referencing **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)**:

**Interpretability: Low.** Like t-SNE, UMAP axes have no unit or meaning. Unlike t-SNE, inter-cluster distances are more meaningful — but still not precisely interpretable.

**Complexity: Medium.** Faster than t-SNE. Approximate nearest neighbor search makes it O(n log n) with smaller constants. Memory-efficient with sparse graph representation.

**Position:** Low interpretability, medium complexity — the modern default for high-dimensional visualization, a step up from t-SNE in scalability and global structure preservation.

---
## How it learns

UMAP is built on the mathematics of Riemannian geometry and fuzzy set theory, but the intuition is similar to t-SNE.

**Step 1 — Build a high-dimensional graph.** For each point, find its n_neighbors nearest neighbors. Connect each point to its neighbors with edges weighted by distance. This creates a fuzzy topological representation of the data's structure.

**Step 2 — Optimize a low-dimensional layout.** Initialize points in 2D (PCA initialization by default). Then iteratively move points to minimize the cross-entropy between the high-dimensional and low-dimensional fuzzy graphs.

**Step 3 — Project.** The result is a 2D layout where connected high-dimensional neighbors are placed close together and disconnected points are pushed apart.

The key difference from t-SNE: UMAP's global graph structure means that inter-cluster distances in the output are more meaningful — widely separated clusters in UMAP are more likely to be genuinely different in the original space.

---
## The math behind it

UMAP models the data's structure as a **fuzzy simplicial set**: a graph where each edge has a weight representing the probability of being a true neighbor.

High-dimensional weight for edge $(i, j)$:
$$w(i, j) = \exp\left(\frac{-d(i,j) - \rho_i}{\sigma_i}\right)$$
Where $d(i,j)$ is the distance, $\rho_i$ is the distance to the nearest neighbor of $i$, and $\sigma_i$ is chosen to normalize the total weight.

Low-dimensional weight:
$$w'(i, j) = \left(1 + a \cdot \|y_i - y_j\|^{2b}\right)^{-1}$$
Where $a$ and $b$ are determined by min_dist.

**Cross-entropy loss** (minimized by gradient descent):
$$C = \sum_{e \in E} w_e \log\frac{w_e}{w'_e} + (1 - w_e) \log\frac{1 - w_e}{1 - w'_e}$$

The cross-entropy balances attractive forces (pull neighbors together) and repulsive forces (push non-neighbors apart), preserving both local and global structure.

---
## Try it yourself

In [ ]:
out1 = Output()

n_neighbors_slider = IntSlider(
    value=15, min=5, max=50, step=5,
    description="n_neighbors:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
min_dist_slider = FloatSlider(
    value=0.1, min=0.0, max=0.5, step=0.05,
    description="min_dist:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)

X, y = load_california_housing()
X_sc = StandardScaler().fit_transform(X)
price_tier = pd.qcut(y, q=3, labels=["Low", "Mid", "High"])

rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_sc), size=1500, replace=False)
X_sample = X_sc[sample_idx]
tier_sample = price_tier.iloc[sample_idx].reset_index(drop=True)

tier_colors = {"Low": PALETTE["primary"], "Mid": PALETTE["accent"], "High": PALETTE["secondary"]}

def render_umap_params(n_neighbors, min_dist):
    if not UMAP_AVAILABLE:
        Z = TSNE(n_components=2, perplexity=30, random_state=42, init="pca").fit_transform(X_sample)
        note = " — umap-learn not installed, showing t-SNE instead (pip install umap-learn)"
    else:
        reducer = umap.UMAP(n_components=2, n_neighbors=n_neighbors, min_dist=min_dist, random_state=42)
        Z = reducer.fit_transform(X_sample)
        note = ""
    traces = []
    for tier, color in tier_colors.items():
        mask = (tier_sample == tier).to_numpy()
        traces.append(go.Scatter(
            x=Z[mask, 0], y=Z[mask, 1], mode="markers",
            marker=dict(color=color, size=5, opacity=0.6),
            name=f"{tier} price",
        ))
    layout = base_layout(
        title=f"UMAP Projection — n_neighbors={n_neighbors}, min_dist={min_dist:.2f}{note}",
        xaxis_title="UMAP dim 1",
        yaxis_title="UMAP dim 2",
    )
    fig = go.Figure(data=traces, layout=layout)

    neighbor_note = (
        "small n_neighbors favors tight, disconnected local clusters" if n_neighbors <= 10
        else "large n_neighbors smooths toward a more connected global shape" if n_neighbors >= 40
        else "a middle-ground setting, balancing local detail with global shape"
    )
    dist_note = (
        "low min_dist packs points close within each cluster" if min_dist <= 0.1
        else "higher min_dist spreads points out, trading tightness for a less crowded view"
    )
    caption = f"**n_neighbors={n_neighbors}, min_dist={min_dist:.2f}.** {neighbor_note.capitalize()}; {dist_note}."

    with out1:
        clear_output(wait=True)
        fig.show()
        display(Markdown(caption))

def on_change_umap_params(change):
    render_umap_params(n_neighbors_slider.value, min_dist_slider.value)

n_neighbors_slider.observe(on_change_umap_params, names="value")
min_dist_slider.observe(on_change_umap_params, names="value")
display(VBox([n_neighbors_slider, min_dist_slider, out1]))
render_umap_params(n_neighbors_slider.value, min_dist_slider.value)

In [ ]:
out2 = Output()

color_dropdown = Dropdown(
    options=[("Price tier", "tier"), ("Median income", "income"), ("Latitude", "lat")],
    value="tier",
    description="Color by:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)

Z_pca3 = PCA(n_components=2, random_state=42).fit_transform(X_sample)
Z_tsne3 = TSNE(n_components=2, perplexity=30, random_state=42, init="pca").fit_transform(X_sample)
if UMAP_AVAILABLE:
    Z_umap3 = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_sample)
    umap_title = "UMAP 2D Projection"
else:
    Z_umap3 = Z_tsne3.copy()
    umap_title = "UMAP (unavailable — showing t-SNE)"

income_sample = X.iloc[sample_idx]["MedInc"].to_numpy()
lat_sample = X.iloc[sample_idx]["Latitude"].to_numpy()

def make_traces3(Z, values_key, show_legend_or_bar):
    if values_key == "tier":
        return [
            go.Scatter(
                x=Z[(tier_sample == t).to_numpy(), 0], y=Z[(tier_sample == t).to_numpy(), 1],
                mode="markers", marker=dict(color=c, size=4, opacity=0.6),
                name=f"{t} price", showlegend=show_legend_or_bar,
            )
            for t, c in tier_colors.items()
        ]
    vals = income_sample if values_key == "income" else lat_sample
    label = "Median income" if values_key == "income" else "Latitude"
    marker = dict(color=vals, colorscale="Viridis", size=4, opacity=0.7, showscale=show_legend_or_bar)
    if show_legend_or_bar:
        marker["colorbar"] = dict(title=label)
    return [go.Scatter(x=Z[:, 0], y=Z[:, 1], mode="markers", marker=marker, name=label, showlegend=False)]

CAPTIONS3 = {
    "tier": "**Coloring by price tier.** All three methods struggle to separate the tiers — this echoes what 07_pca and 08_tsne already showed: price is not the dominant signal in this dataset's geometry.",
    "income": "**Coloring by median income.** Income tracks this structure more directly than price tier did — compare how smoothly it varies across each panel.",
    "lat": "**Coloring by latitude.** The gradient largely reflects Northern vs. Southern California — geography is one of the strongest signals in this dataset.",
}

def render_three_way(values_key):
    fig = make_subplots(rows=1, cols=3, subplot_titles=("PCA 2D Projection", "t-SNE 2D Projection", umap_title))
    for col, Z in enumerate([Z_pca3, Z_tsne3, Z_umap3], start=1):
        show = (col == 3)
        for trace in make_traces3(Z, values_key, show):
            fig.add_trace(trace, row=1, col=col)
    fig.update_layout(
        base_layout(title="PCA vs. t-SNE vs. UMAP", xaxis_title="", yaxis_title=""),
        height=420, showlegend=(values_key == "tier"),
    )
    with out2:
        clear_output(wait=True)
        fig.show()
        display(Markdown(CAPTIONS3[values_key]))

def on_change_three_way(change):
    render_three_way(color_dropdown.value)

color_dropdown.observe(on_change_three_way, names="value")
display(VBox([color_dropdown, out2]))
render_three_way(color_dropdown.value)

---
## What's happening?

The n_neighbors widget shows UMAP's key tradeoff: small n_neighbors focuses on immediate local structure, creating distinct tight clusters but potentially losing the big picture. Large n_neighbors considers broader context, smoothing local detail in favor of global organization.

The three-way comparison shows why UMAP has become the default choice: it combines t-SNE's ability to reveal non-linear cluster structure with better preservation of global relationships and significantly faster computation. The clusters it produces are more stable between runs (UMAP with fixed random_state is deterministic) and inter-cluster distances carry more meaning than in t-SNE. Try switching the coloring dropdown to median income or latitude to see which factor each method's layout is actually tracking.

---
## Key hyperparameters

**`n_neighbors`** (default 15): Controls local vs global structure. Low (5-10): tight local clusters. High (50+): global topology. Think of it as equivalent to t-SNE's perplexity.

**`min_dist`** (default 0.1): Minimum distance between points in the 2D layout. Low: tighter clusters (better for visualization). High: more spread (better for preserving density).

**`n_components`** (default 2): Dimensions of the output. Can be set to 3 for 3D visualization or higher for feature engineering.

umap-learn docs: [umap-learn.readthedocs.io](https://umap-learn.readthedocs.io/en/latest/)

---
## Strengths and weaknesses

| Strength | Weakness |
|----------|----------|
| Faster than t-SNE (5-10x) | More complex mathematical foundation |
| Preserves global structure better than t-SNE | Still stochastic without fixed random_state |
| Can project new data points without rerunning | Requires umap-learn (not in sklearn) |
| Scales to millions of samples | n_neighbors and min_dist require tuning |
| Better for feature engineering (not just viz) | Less published validation than t-SNE |

---
## When to use it / When NOT to use it

| Use UMAP when... | Do NOT use UMAP when... |
|------------------|--------------------------|
| t-SNE is too slow | umap-learn is not available |
| You need to project new points | You need a pure sklearn solution |
| Global cluster distances matter | Dataset is tiny (< 50 points) |
| You want reproducible results (set random_state) | You need a fully explainable algorithm |
| Feature engineering (not just visualization) | Perfect local separation is more important than speed |

---
## Key takeaway

> **UMAP is the modern default for non-linear dimensionality reduction: faster than t-SNE, better global structure, and able to project new data — but like all DR methods, it reveals structure rather than measuring it.**

---
*Next up: 10_dimensionality_reduction_comparison — where all three methods meet on the same real dataset*